In [3]:
%pip install gensim
%pip install imbalanced-learn
%pip install pandas

  Using cached gensim-4.4.0.tar.gz (23.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-2.1.1-cp314-cp314-win_amd64.whl.metadata (7.6 kB)
Using cached smart_open-7.5.1-py3-none-any.whl (64 kB)
Using cached wrapt-2.1.1-cp314-cp314-win_amd64.whl (60 kB)
Failed to build gensim
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [716 lines of output]
      C:\Users\nauja\AppData\Local\Temp\pip-build-env-ac036bse\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\nauja\AppData\Local\Temp\pip-build-env-ac036bse\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\gensim
      copying gensim\downloader.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\interfaces.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\matutils.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\nosy.py -> build\lib.win-amd64-cpython-314\gensim
   

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
df = pd.read_csv("cleaned_file.csv", on_bad_lines='skip')

df.head()

,payload,classification
0,sullivanjj@state.gov,high
1,"Koh, Harold Hongju",high
2,"Dear customer, you have 16,590 points which wi...",low
3,mark.guzman@enron.com,high
4,"icd_code=0090, icd_version=9, diagnosis=Infect...",high


In [5]:
df = df.rename(columns={
    df.columns[0]: 'Tag',
    df.columns[1]: 'Sensitivity'
})

df['Sensitivity'] = (
    df['Sensitivity']
        .astype(str)          # ensure string
        .str.strip()          # remove spaces
        .str.capitalize()     # High / Low format
)




In [6]:
print(df['Sensitivity'].value_counts())

Sensitivity
High    5876
Low     4425
Name: count, dtype: int64


In [7]:
print(df['Sensitivity'].unique())
print(df['Tag'].nunique())


<StringArray>
['High', 'Low']
Length: 2, dtype: str
9393


In [8]:
df = df.dropna(subset=['Sensitivity'])
df = df.drop_duplicates(subset='Tag', keep=False)

df.shape[0]

print(df['Sensitivity'].unique())
print(df['Tag'].nunique())


<StringArray>
['High', 'Low']
Length: 2, dtype: str
9006


In [9]:
import re


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '<NUM>', text)           # Replace all numbers
    text = re.sub(r'[=;:/\\._-]|', ' ', text)       # Tokenize on symbols
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [10]:
from sklearn.model_selection import train_test_split

X = df['Tag']
y = df['Sensitivity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (7204,)
Shape of X_test: (1802,)
Shape of y_train: (7204,)
Shape of y_test: (1802,)


In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import FunctionTransformer
from imblearn.over_sampling import SMOTE


count_vectorizer = CountVectorizer(
    analyzer='char_wb',
    ngram_range=(3,5),
    max_features=5000,
    min_df=2,
    preprocessor=preprocess_text
)

# Vectorization and SMOTE are done outside of the pipeline for fitting on the entire dataset
X_vectorized = count_vectorizer.fit_transform(df['Tag'])

smote = SMOTE(random_state=42) #try with no oversampling
X_resampled, y_resampled = smote.fit_resample(X_vectorized, df['Sensitivity'])

print(pd.Series(y_resampled).value_counts())


clf_pipeline_cv = Pipeline([
    ("count_vectorizer", count_vectorizer),
    ("clf", DecisionTreeClassifier(random_state=38))
])

clf_pipeline_cv.fit(X_train, y_train)

y_pred_cv = clf_pipeline_cv.predict(X_test)
print(classification_report(y_test, y_pred_cv))

dt_f1_macro = f1_score(y_test, y_pred_cv, average='macro')
dt_accuracy = accuracy_score(y_test, y_pred_cv)
print(f"Decision Tree Macro F1-score: {dt_f1_macro:.6f}")
print(f"Decision Tree Accuracy: {dt_accuracy:.6f}")

Sensitivity
High    4875
Low     4875
Name: count, dtype: int64
              precision    recall  f1-score   support

        High       0.98      0.99      0.99       975
         Low       0.99      0.98      0.99       827

    accuracy                           0.99      1802
   macro avg       0.99      0.99      0.99      1802
weighted avg       0.99      0.99      0.99      1802

Decision Tree Macro F1-score: 0.986583
Decision Tree Accuracy: 0.986681


In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.decomposition import TruncatedSVD


clf_pipeline_knn = Pipeline([
    ("count_vectorizer", count_vectorizer),
    ("svd", TruncatedSVD(n_components=15, random_state=42)),
    ("clf", KNeighborsClassifier(n_neighbors=10))
])

clf_pipeline_knn.fit(X_train, y_train)
y_pred_knn = clf_pipeline_knn.predict(X_test)
print("=== K-Nearest Neighbors ===")
print(classification_report(y_test, y_pred_knn))

knn_f1_macro = f1_score(y_test, y_pred_knn, average='macro')
knn_accuracy = accuracy_score(y_test, y_pred_knn)
print(f"KNN Macro F1-score: {knn_f1_macro:.6f}")
print(f"KNN Accuracy: {knn_accuracy:.6f}")

=== K-Nearest Neighbors ===
              precision    recall  f1-score   support

        High       0.96      0.98      0.97       975
         Low       0.98      0.95      0.97       827

    accuracy                           0.97      1802
   macro avg       0.97      0.97      0.97      1802
weighted avg       0.97      0.97      0.97      1802

KNN Macro F1-score: 0.969764
KNN Accuracy: 0.970033


In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, accuracy_score

clf_pipeline_nb = Pipeline([
    ("count_vectorizer", count_vectorizer),
    ("clf", MultinomialNB())
])

clf_pipeline_nb.fit(X_train, y_train)
y_pred_nb = clf_pipeline_nb.predict(X_test)
print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))

nb_f1_macro = f1_score(y_test, y_pred_nb, average='macro')
nb_accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Naive Bayes Macro F1-score: {nb_f1_macro:.6f}")
print(f"Naive Bayes Accuracy: {nb_accuracy:.6f}")

=== Naive Bayes ===
              precision    recall  f1-score   support

        High       0.74      0.93      0.83       975
         Low       0.89      0.61      0.73       827

    accuracy                           0.79      1802
   macro avg       0.81      0.77      0.78      1802
weighted avg       0.81      0.79      0.78      1802

Naive Bayes Macro F1-score: 0.775743
Naive Bayes Accuracy: 0.786903


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from datetime import datetime


param_grid_dt = {
    "clf__criterion": ["gini", "entropy", "log_loss"],
    "clf__max_depth": [10,11,12,13],
    "clf__min_samples_split": [2,3,4,5],
    "clf__min_samples_leaf": [2,3,4],
    "count_vectorizer__ngram_range": [(1, 2), (2, 3), (3, 4), (3,5)],
}

grid_search = GridSearchCV(
    clf_pipeline_cv,
    param_grid=param_grid_dt,
    cv=8,
    scoring="f1_macro",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score: ", grid_search.best_score_)
print("\nClassification Report for the best model:")
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))

accuracy = accuracy_score(y_test, y_pred_best)
print(f"\nAccuracy: {accuracy}")

with open("TuningLogs.txt", "a") as f:
    f.write("===== DT Grid Search Results =====\n")
    f.write(f"Time: {datetime.now()}\n")
    f.write(f"Grid: {param_grid_dt}\n")
    f.write(f"Best Params: {grid_search.best_params_}\n")
    f.write(f"Best Score: {grid_search.best_score_}\n\n")


In [ ]:
param_grid_knn = {
    "svd__n_components": [10, 13, 15],  # all must be <16
    "clf__n_neighbors": [11,12,13,14],
    "clf__weights": ["uniform", "distance"],
    "clf__metric": ["euclidean", "manhattan"],
    "count_vectorizer__ngram_range": [(2,3), (3,4), (3,5)],
}

grid_knn = GridSearchCV(
    clf_pipeline_knn,
    param_grid=param_grid_knn,
    cv=8,
    scoring="f1_macro",
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)
best_model = grid_knn.best_estimator_
print("Best KNN params:", grid_knn.best_params_)
print("Best CV F1 Score:", grid_knn.best_score_)
print("\nClassification Report for the best model:")
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))

accuracy = accuracy_score(y_test, y_pred_best)
print(f"\nAccuracy: {accuracy}")

with open("TuningLogs.txt", "a") as f:
    f.write("===== KNN Grid Search Results =====\n")
    f.write(f"Grid: {param_grid_knn}\n")
    f.write(f"Best Params: {grid_knn.best_params_}\n")
    f.write(f"Best Score: {grid_knn.best_score_}\n\n")



Best KNN params: {'clf__metric': 'euclidean', 'clf__n_neighbors': 12, 'clf__weights': 'distance', 'count_vectorizer__ngram_range': (2, 3), 'svd__n_components': 15}
Best CV F1 Score: 0.9734336702292117

Classification Report for the best model:
              precision    recall  f1-score   support

        High       0.97      0.98      0.97       975
         Low       0.98      0.96      0.97       827

    accuracy                           0.97      1802
   macro avg       0.97      0.97      0.97      1802
weighted avg       0.97      0.97      0.97      1802


Accuracy: 0.9716981132075472


In [ ]:
param_grid_nb = {
    "clf__alpha": [0.3,0.1,0.01],
    "clf__fit_prior": [True, False],
    "clf__force_alpha": [True, False],
    "count_vectorizer__ngram_range": [(1, 2), (2, 3), (3, 4), (4,5)]

}

grid_nb = GridSearchCV(
    clf_pipeline_nb,
    param_grid=param_grid_nb,
    cv=8,
    scoring="f1_macro",
    n_jobs=-1
)

grid_nb.fit(X_train, y_train)
best_model = grid_nb.best_estimator_

print("Best Naive Bayes params:", grid_nb.best_params_)
print("Best CV accuracy:", grid_nb.best_score_)
print("\nClassification Report for the best model:")
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))

accuracy = accuracy_score(y_test, y_pred_best)
print(f"\nAccuracy: {accuracy}")

with open("TuningLogs.txt", "a") as f:
    f.write("===== NB Grid Search Results =====\n")
    f.write(f"Grid: {param_grid_nb}\n")

    f.write(f"Best Params: {grid_nb.best_params_}\n")
    f.write(f"Best Score: {grid_nb.best_score_}\n\n")



Best Naive Bayes params: {'clf__alpha': 0.01, 'clf__fit_prior': True, 'clf__force_alpha': True, 'count_vectorizer__ngram_range': (3, 4)}
Best CV accuracy: 0.7626563226572904

Classification Report for the best model:
              precision    recall  f1-score   support

        High       0.74      0.94      0.83       975
         Low       0.90      0.61      0.73       827

    accuracy                           0.79      1802
   macro avg       0.82      0.78      0.78      1802
weighted avg       0.81      0.79      0.78      1802


Accuracy: 0.7913429522752498


In [ ]:
import joblib

# The best performing model was the KNN model based on the GridSearchCV results.
best_model_to_export = grid_knn.best_estimator_

# Define the filename for the exported model
model_filename = 'best_knn_model.joblib'

# Export the model
joblib.dump(best_model_to_export, model_filename)

print(f"Best model exported to {model_filename}")

Best model exported to best_knn_model.joblib


In [ ]:
model2 = grid_search.best_estimator_

model_filename = 'dt.joblib'

#joblib.dump(model2, model_filename)

#print(f"Best model exported to {model_filename}")

model3 = grid_nb.best_estimator_

model_filename = 'nb.joblib'

#joblib.dump(model3, model_filename)

In [ ]:
# ==========================================
# FINAL EVALUATION CELL
# Includes Decision Tree, KNN, Naive Bayes, AND AI MODEL
# ==========================================

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import precision_score, recall_score
import numpy as np
import pandas as pd
import time

# --- AI IMPORT ---
from gpt4_classification import classify_batch_with_ai

print("STARTING COMPREHENSIVE EVALUATION (INCLUDING AI MODEL)...")

X_full = df['Tag']
y_full = df['Sensitivity']

dt_model = grid_search.best_estimator_
knn_model = grid_knn.best_estimator_
nb_model = grid_nb.best_estimator_

print("=" * 70)
print("3-Fold Cross-Validation with Different Train-Test Splits")
print("Using the FULL dataset with 3 different 80-20 splits")
print("=" * 70)

sss = StratifiedShuffleSplit(
    n_splits=3,        # run 3 times
    test_size=0.2,     # 20% test
    train_size=0.8,    # 80% train
    random_state=42
)

# Initialize results dictionary including AI Model
results = {
    'Decision Tree': {'accuracy': [], 'f1_macro': [], 'precision': [], 'recall': []},
    'KNN': {'accuracy': [], 'f1_macro': [], 'precision': [], 'recall': []},
    'Naive Bayes': {'accuracy': [], 'f1_macro': [], 'precision': [], 'recall': []},
    'AI Model': {'accuracy': [], 'f1_macro': [], 'precision': [], 'recall': []}
}

for fold_num, (train_idx, test_idx) in enumerate(sss.split(X_full, y_full), 1):
    print(f"\n{'='*70}")
    print(f"SPLIT {fold_num} - Training on {len(train_idx)} samples, Testing on {len(test_idx)} samples")
    print(f"{'='*70}")
    
    X_train_fold = X_full.iloc[train_idx]
    X_test_fold = X_full.iloc[test_idx]
    y_train_fold = y_full.iloc[train_idx]
    y_test_fold = y_full.iloc[test_idx]

    print("Shape of X_train:", X_train_fold.shape)
    print("Shape of X_test:", X_test_fold.shape)
    print("Shape of y_train:", y_train_fold.shape)
    print("Shape of y_test:", y_test_fold.shape)
    
    print("\n--- Decision Tree ---")
    dt_model.fit(X_train_fold, y_train_fold)
    y_pred_dt = dt_model.predict(X_test_fold)
    dt_acc = accuracy_score(y_test_fold, y_pred_dt)
    dt_f1 = f1_score(y_test_fold, y_pred_dt, average='macro')
    dt_precision = precision_score(y_test_fold, y_pred_dt, average='macro')
    dt_recall = recall_score(y_test_fold, y_pred_dt, average='macro')
    
    results['Decision Tree']['accuracy'].append(dt_acc)
    results['Decision Tree']['f1_macro'].append(dt_f1)
    results['Decision Tree']['precision'].append(dt_precision)
    results['Decision Tree']['recall'].append(dt_recall)
    
    print(f"Accuracy: {dt_acc:.4f}, F1: {dt_f1:.4f}, Precision: {dt_precision:.4f}, Recall: {dt_recall:.4f}")
    
    print("\n--- K-Nearest Neighbors ---")
    knn_model.fit(X_train_fold, y_train_fold)
    y_pred_knn = knn_model.predict(X_test_fold)
    knn_acc = accuracy_score(y_test_fold, y_pred_knn)
    knn_f1 = f1_score(y_test_fold, y_pred_knn, average='macro')
    knn_precision = precision_score(y_test_fold, y_pred_knn, average='macro')
    knn_recall = recall_score(y_test_fold, y_pred_knn, average='macro')
    
    results['KNN']['accuracy'].append(knn_acc)
    results['KNN']['f1_macro'].append(knn_f1)
    results['KNN']['precision'].append(knn_precision)
    results['KNN']['recall'].append(knn_recall)
    
    print(f"Accuracy: {knn_acc:.4f}, F1: {knn_f1:.4f}, Precision: {knn_precision:.4f}, Recall: {knn_recall:.4f}")
    
    print("\n--- Naive Bayes ---")
    nb_model.fit(X_train_fold, y_train_fold)
    y_pred_nb = nb_model.predict(X_test_fold)
    nb_acc = accuracy_score(y_test_fold, y_pred_nb)
    nb_f1 = f1_score(y_test_fold, y_pred_nb, average='macro')
    nb_precision = precision_score(y_test_fold, y_pred_nb, average='macro')
    nb_recall = recall_score(y_test_fold, y_pred_nb, average='macro')
    
    results['Naive Bayes']['accuracy'].append(nb_acc)
    results['Naive Bayes']['f1_macro'].append(nb_f1)
    results['Naive Bayes']['precision'].append(nb_precision)
    results['Naive Bayes']['recall'].append(nb_recall)
    
    print(f"Accuracy: {nb_acc:.4f}, F1: {nb_f1:.4f}, Precision: {nb_precision:.4f}, Recall: {nb_recall:.4f}")

    # --- AI Model Integration ---
    print("\n--- AI Model (Batch Processing) ---")
    # We need to process the fold in batches to respect the token limits/design
    ai_preds = []
    
    # Convert series to list for batching
    X_test_list = X_test_fold.tolist()
    
    BATCH_SIZE = 50
    total_ai = len(X_test_list)
    
    for i in range(0, total_ai, BATCH_SIZE):
        batch_texts = X_test_list[i:i+BATCH_SIZE]
        print(f"Processing AI batch {i // BATCH_SIZE + 1}...", end='\r')
        # CALLING THE IMPORTED FUNCTION
        batch_p, _, _ = classify_batch_with_ai(batch_texts)
        ai_preds.extend(batch_p)
    
    print("AI Classification complete.             ")
    
    # Calculate metrics for AI
    ai_acc = accuracy_score(y_test_fold, ai_preds)
    ai_f1 = f1_score(y_test_fold, ai_preds, average='macro')
    ai_precision = precision_score(y_test_fold, ai_preds, average='macro')
    ai_recall = recall_score(y_test_fold, ai_preds, average='macro')
    
    results['AI Model']['accuracy'].append(ai_acc)
    results['AI Model']['f1_macro'].append(ai_f1)
    results['AI Model']['precision'].append(ai_precision)
    results['AI Model']['recall'].append(ai_recall)
    
    print(f"Accuracy: {ai_acc:.4f}, F1: {ai_f1:.4f}, Precision: {ai_precision:.4f}, Recall: {ai_recall:.4f}")

    
    print(f"\n--- Split {fold_num} Summary ---")
    split_comparison = pd.DataFrame({
        'Model': ['Decision Tree', 'KNN', 'Naive Bayes', 'AI Model'],
        'Accuracy': [dt_acc, knn_acc, nb_acc, ai_acc],
        'F1-Score': [dt_f1, knn_f1, nb_f1, ai_f1],
        'Precision': [dt_precision, knn_precision, nb_precision, ai_precision],
        'Recall': [dt_recall, knn_recall, nb_recall, ai_recall]
    })
    print(split_comparison.to_string(index=False))
    best_model = split_comparison.loc[split_comparison['Accuracy'].idxmax(), 'Model']
    print(f"\nBest model for Split {fold_num}: {best_model}")


print("\n" + "=" * 70)
print("COMPREHENSIVE SUMMARY - All Train-Test Splits")
print("=" * 70)

# Per-split comparison table
summary_data = []
for fold_num in range(1, 4):
    summary_data.append({
        'Split': fold_num,
        'DT_Acc': f"{results['Decision Tree']['accuracy'][fold_num-1]:.4f}",
        'DT_F1': f"{results['Decision Tree']['f1_macro'][fold_num-1]:.4f}",
        'KNN_Acc': f"{results['KNN']['accuracy'][fold_num-1]:.4f}",
        'KNN_F1': f"{results['KNN']['f1_macro'][fold_num-1]:.4f}",
        'NB_Acc': f"{results['Naive Bayes']['accuracy'][fold_num-1]:.4f}",
        'NB_F1': f"{results['Naive Bayes']['f1_macro'][fold_num-1]:.4f}",
        'AI_Acc': f"{results['AI Model']['accuracy'][fold_num-1]:.4f}",
        'AI_F1': f"{results['AI Model']['f1_macro'][fold_num-1]:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\nPer-Split Performance:")
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("OVERALL STATISTICS (Mean ± Std)")
print("=" * 70)

stats_data = []
for model_name in ['Decision Tree', 'KNN', 'Naive Bayes', 'AI Model']:
    acc_mean = np.mean(results[model_name]['accuracy'])
    acc_std = np.std(results[model_name]['accuracy'])
    f1_mean = np.mean(results[model_name]['f1_macro'])
    f1_std = np.std(results[model_name]['f1_macro'])
    prec_mean = np.mean(results[model_name]['precision'])
    prec_std = np.std(results[model_name]['precision'])
    rec_mean = np.mean(results[model_name]['recall'])
    rec_std = np.std(results[model_name]['recall'])
    
    stats_data.append({
        'Model': model_name,
        'Accuracy': f"{acc_mean:.4f} ± {acc_std:.4f}",
        'F1-Score': f"{f1_mean:.4f} ± {f1_std:.4f}",
        'Precision': f"{prec_mean:.4f} ± {prec_std:.4f}",
        'Recall': f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

stats_df = pd.DataFrame(stats_data)
print(stats_df.to_string(index=False))

best_overall = max(
    ['Decision Tree', 'KNN', 'Naive Bayes', 'AI Model'],
    key=lambda x: np.mean(results[x]['f1_macro'])
)
print(f"\n🏆 Best Overall Model (by mean F1-score): {best_overall}")

5-Fold Cross-Validation with Different Train-Test Splits
Using the FULL dataset with 5 different 80-20 splits

SPLIT 1 - Training on 7204 samples, Testing on 1802 samples
Shape of X_train: (7204,)
Shape of X_test: (1802,)
Shape of y_train: (7204,)
Shape of y_test: (1802,)

--- Decision Tree ---
Accuracy: 0.9872, F1: 0.9872, Precision: 0.9867, Recall: 0.9877
              precision    recall  f1-score   support

        High       0.99      0.98      0.99       975
         Low       0.98      0.99      0.99       827

    accuracy                           0.99      1802
   macro avg       0.99      0.99      0.99      1802
weighted avg       0.99      0.99      0.99      1802


--- K-Nearest Neighbors ---
Accuracy: 0.9717, F1: 0.9715, Precision: 0.9720, Recall: 0.9710
              precision    recall  f1-score   support

        High       0.97      0.98      0.97       975
         Low       0.98      0.96      0.97       827

    accuracy                           0.97      1802
  